# Bài học 14: Writer Agent và Image Agent

Ở bài học trước, chúng ta đã xây dựng Research Agent và Outline Agent. Bây giờ, ta sẽ xây 2 agent cuối cùng:

3. **Writer Agent** — Viết toàn bộ bài viết từ dàn ý
4. **Image Agent** — Tìm và chèn hình ảnh (tùy chọn)

```
ContentOutline --> [Writer Agent] --> Markdown article --> [Image Agent] --> Enriched article
```

Điểm khác biệt quan trọng: Writer Agent sử dụng **Grok-4** (xAI), không phải Claude. Ở Bài học 7, chúng ta đã thảo luận tại sao các model khác nhau được chọn cho các tác vụ khác nhau — đây chính là quyết định đó trong thực tế.

## Trước tiên: Markdown là gì?

Writer Agent tạo ra bài viết ở định dạng **Markdown** — một định dạng văn bản đơn giản dùng để tạo tài liệu có định dạng. Tất cả file trong thư mục `content/` đều là file Markdown (`.md`).

Markdown hoạt động như sau:

| Bạn gõ | Kết quả hiển thị |
|---|---|
| `# Title` | Tiêu đề lớn (H1) |
| `## Section` | Tiêu đề vừa (H2) |
| `### Subsection` | Tiêu đề nhỏ (H3) |
| `**bold text**` | **bold text** |
| `- item` | Dấu chấm đầu dòng |
| `1. item` | Danh sách đánh số |
| `[link text](url)` | Liên kết bấm được |
| `![alt text](image-url)` | Một hình ảnh |

**Tại sao dùng Markdown?**
- Là văn bản thuần — dễ lưu trữ, quản lý phiên bản, và xử lý bằng code
- Chuyển sang HTML cho website chỉ bằng một lệnh
- Là định dạng tiêu chuẩn cho các hệ thống quản lý nội dung
- Bạn đang đọc Markdown ngay lúc này — notebook này dùng Markdown cho các ô văn bản!

## Tại sao dùng Grok cho Writer?

Ở Bài học 7, chúng ta đã thảo luận về việc các model khác nhau có thế mạnh khác nhau. Đây là quyết định đó trong thực tế:

| Agent | Model | Lý do |
|-------|-------|--------|
| Research | Claude Sonnet | Cần tools (tìm kiếm web) + tư duy có cấu trúc |
| Outline | Claude Sonnet | Cần output_schema (JSON) |
| **Writer** | **Grok-4** | **Phong cách viết tự nhiên, giọng văn linh hoạt** |
| Image | Claude Sonnet | Cần tools (gọi API) + output_schema |

### Hạn chế của Grok

Grok có một **hạn chế quan trọng**: nó không thể sử dụng `tools` và `output_schema` cùng lúc. Do đó:
- Writer Agent **không có tools** — nó chỉ nhận văn bản và viết
- Writer Agent **không có output_schema** — nó trả về Markdown thuần trực tiếp

Claude hỗ trợ cả hai cùng lúc, đó là lý do Research Agent và Outline Agent có thể gộp thành một agent (nhưng chúng ta giữ tách biệt để code rõ ràng và dễ bảo trì hơn).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.xai import xAI

writer_agent = Agent(
    name="Writer Agent",
    model=xAI(id="grok-4"),
    instructions=[
        "You are an expert SEO content writer.",
        "Write a comprehensive, SEO-optimized article following the provided outline exactly.",
        "Use proper Markdown: H1 for title, H2 for sections, H3 for subheadings.",
        "Bold important keywords naturally.",
        "Aim for 1500-2500 words.",
        "Do NOT output anything other than the article Markdown.",
    ],
    markdown=True,
)

## Thử nghiệm Writer Agent

Chúng ta sẽ truyền một **dàn ý mẫu** (dưới dạng chuỗi JSON) cho Writer Agent. Trong sản phẩm thực tế, dàn ý này đến từ Outline Agent ở bước trước.

Writer Agent sẽ:
- Đọc dàn ý và hiểu cấu trúc
- Viết một bài viết hoàn chỉnh bằng Markdown
- Đảm bảo mọi phần trong dàn ý đều được đề cập
- Lồng ghép từ khóa một cách tự nhiên xuyên suốt bài viết

> **Chi phí:** ~$0.50-1.00 (Grok-4 viết một bài dài). Mất 1-2 phút.

In [ ]:
sample_outline = """{
  "title": "On-Page SEO Guide 2026",
  "meta_description": "Learn how to optimize on-page SEO to boost your website rankings.",
  "target_keywords": ["on-page SEO", "SEO optimization"],
  "sections": [
    {"heading": "What Is On-Page SEO?", "key_points": ["Definition", "Why it matters"]},
    {"heading": "Optimizing Title Tags", "key_points": ["Ideal length", "Keyword placement"]},
    {"heading": "Conclusion", "key_points": ["Summary", "Call to action"]}
  ],
  "tone": "informative"
}"""

print("Đang viết bài từ dàn ý...")
response = writer_agent.run(f"Write a full SEO article based on this outline:\n\n{sample_outline}")
article = response.content
print(f"Xong! {len(article.split())} từ\n")
print(article[:1000] + "\n...")

## Image Agent (Tùy chọn)

Agent cuối cùng trong pipeline. Nhiệm vụ của nó:
- Đọc bài viết và xác định vị trí nào cần hình ảnh
- Tìm kiếm hình ảnh phù hợp qua **Freepik API** hoặc **DataForSEO API**
- Chèn hình ảnh vào bài viết kèm alt text thân thiện SEO

### Tại sao là tùy chọn?

Image Agent cần API key từ Freepik hoặc DataForSEO. Nếu bạn không có:
- Pipeline **bỏ qua** bước này hoàn toàn
- Bài viết vẫn được tạo bình thường, chỉ là không có hình ảnh
- Không có lỗi nào xảy ra

Đây là thiết kế có chủ đích — sản phẩm vẫn hoạt động bình thường ngay cả khi không mua API key cho hình ảnh.

In [ ]:
import os
from agno.models.anthropic import Claude

# Minh họa khái niệm (không chạy thật nếu chưa có key)
print("Image Agent là tùy chọn:")
print(f"  FREEPIK_API_KEY: {'Đã cài' if os.getenv('FREEPIK_API_KEY') else 'Chưa cài'}")
print(f"  DATA_FOR_SEO_API_KEY: {'Đã cài' if os.getenv('DATA_FOR_SEO_API_KEY') else 'Chưa cài'}")
print()
print("Nếu không có API key, pipeline sẽ bỏ qua bước này.")
print("Bài viết vẫn được tạo bình thường, chỉ là không có hình ảnh.")

## Tổng Kết

Chúng ta đã xây dựng xong cả **4 agent** cho SEO pipeline:

```
Topic
  |-> [Research Agent]  -- Claude Sonnet + DuckDuckGo    --> Research notes
  |-> [Outline Agent]   -- Claude Sonnet + output_schema --> ContentOutline (JSON)
  |-> [Writer Agent]    -- Grok-4, plain Markdown        --> Full article
  |-> [Image Agent]     -- Claude Sonnet + image APIs    --> Enriched article (optional)
```

**Những điểm chính:**
- Chọn đúng model cho từng tác vụ (Claude cho tools/schema, Grok cho viết bài) — đây là khung quyết định từ Bài học 7
- Hạn chế của Grok: không thể dùng tools + output_schema cùng lúc
- Thiết kế tính năng tùy chọn một cách tinh tế — pipeline vẫn hoạt động ngay cả khi không có API key cho hình ảnh

**Bài học tiếp theo**: Kết nối tất cả agent thành một pipeline hoàn chỉnh có theo dõi trạng thái qua database!

## Bài Tập

Xem biến `article` từ phần thử nghiệm Writer ở trên. Viết code để:

1. Đếm tổng số từ (`len(article.split())`)
2. Đếm có bao nhiêu tiêu đề H2 (`##`) trong bài viết (gợi ý: `article.count("## ")`)
3. Kiểm tra bài viết có chứa từ khóa "on-page SEO" không (gợi ý: `"on-page SEO".lower() in article.lower()`)

Đây là cách bạn kiểm tra chất lượng cơ bản cho nội dung được tạo ra trước khi xuất bản.

In [ ]:
# Bài tập: Viết code của bạn ở đây
